# Lab 6 — Notebook 1: DataFrames, S3A, and Catalyst pushdown

**What you'll do here:** stand up a SparkSession that reads parquet directly from S3, load TPC-H `lineitem`, and observe Catalyst's predicate pushdown and column pruning in the Spark UI. Every DataFrame verb in this notebook appears in the Week 07 deck.

**Before you start:** make sure your AWS Academy Learner Lab is active and your `~/.aws/credentials` file has fresh credentials (the same file you used in Lab 5). The docker-compose.yml mounts that file into the container read-only.


## 2.1 SparkSession configured for S3A

The Docker image we built (see `Dockerfile` in the lab folder) replaces the bundled Hadoop 3.3.4 client with 3.3.6 and adds the `hadoop-aws` + `aws-java-sdk-bundle` JARs directly into `$SPARK_HOME/jars`. That way the SparkSession starts in 5–10 seconds without resolving packages at runtime.

Three settings we still pass at session creation:

- **`spark.hadoop.fs.s3a.aws.credentials.provider`** — pinned to `ProfileCredentialsProvider` so it reads `~/.aws/credentials` (mounted read-only by `docker-compose.yml`) instead of trying the full default provider chain.
- **`spark.hadoop.fs.s3a.requester.pays.enabled = true`** — opt in to Requester Pays. Without it, every S3 read returns 403 from this bucket.
- **`spark.hadoop.fs.s3a.endpoint`** — pinned to `us-east-1` to avoid an extra round-trip for region discovery.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Lab6-N1-DataFrames")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider",
    )
    .config("spark.hadoop.fs.s3a.requester.pays.enabled", "true")
    .config("spark.hadoop.fs.s3a.endpoint", "s3.us-east-1.amazonaws.com")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")
print(f"Spark UI:      http://localhost:4040")


**Expected output:** `Spark version: 3.5.3` followed by the Spark UI URL. Open the URL in another browser tab — you'll come back to it throughout this notebook. Note that this cell builds a SparkSession but doesn't touch S3 yet — that comes next.


## 2.2 Load `lineitem` from S3

TPC-H `lineitem` at scale factor 1 is about 6 million rows / 150 MB of parquet across multiple files, staged in the same S3 bucket you used in Lab 5. The `spark.read.parquet(...)` call below is lazy — it only fetches the schema. The `.show(5)` and `.count()` calls that follow are actions that trigger actual S3 reads.

This is the **first cell that actually touches S3**. If anything is wrong with credentials, network, or Requester Pays opt-in, you'll see it here, not in 2.1.


In [ ]:
LINEITEM_PATH = "s3a://tpch-torstengrabs-parquet/1GB/lineitem/"

lineitem = spark.read.parquet(LINEITEM_PATH)
lineitem.printSchema()
print(f"{lineitem.count():,} rows")
lineitem.show(5)


**Troubleshooting** — if this cell raises an exception, work through these in order:

1. **`AccessDenied` or `400 Bad Request`** — almost always stale AWS credentials. AWS Academy Learner Lab session tokens expire after about 4 hours. Refresh them in the Learner Lab (**AWS Details → AWS CLI → Show**), paste the new block into `~/.aws/credentials`, then **Kernel → Restart Kernel** and re-run 2.1 and 2.2. The credentials are read once at SparkSession creation, so a kernel restart is required for them to take effect.
2. **`com.amazonaws.AmazonClientException: Unable to load AWS credentials`** — your `~/.aws/credentials` isn't visible inside the container. Confirm the file exists on your host, then `docker compose down` and `docker compose up --build` to remount it.
3. **`NoClassDefFoundError: ...PrefetchingStatistics`** — the container is running the base image without our Dockerfile's Hadoop swap. Run `docker compose up --build` (note the `--build` flag).
4. **Network timeouts** — check that you can reach `https://s3.us-east-1.amazonaws.com` from the host (corporate firewalls sometimes block it). If you're on a VPN, try without it.


## 2.3 Exercise — most common ship modes

Write a DataFrame chain that returns the **10 most common values of `l_shipmode`**, with their counts, sorted descending by count. Use `groupBy` → `count` → `orderBy` → `show` (deck slides 28–29).


In [ ]:
# TODO: implement the query described above.
# Hint: lineitem.groupBy(...).count().orderBy(...).show(10)


**Expected output:** 7 rows (TPC-H has exactly 7 distinct ship modes), with counts in the high 800,000s each. The result fits in a tiny aggregation result returned to the driver.


## 2.4 Cache and re-run

Caching pins data in memory across the cluster (deck slide 13). Useful when you'll touch the same intermediate result several times — for example, during interactive exploration.


In [ ]:
import time

filtered = (lineitem
            .select("l_returnflag", "l_linestatus", "l_quantity",
                    "l_extendedprice", "l_discount", "l_shipdate")
            .filter(F.col("l_shipdate") <= "1998-09-02"))

# Trigger the cache by calling an action
filtered.cache()
t = time.time(); n = filtered.count(); print(f"first count (cold):    {n:,} rows in {time.time()-t:.1f}s")
t = time.time(); n = filtered.count(); print(f"second count (cached): {n:,} rows in {time.time()-t:.1f}s")


The second call should be **near-instant** — Spark didn't go back to S3, it read from the in-memory cache. In the Spark UI's **Storage** tab you'll see the cached DataFrame listed with size and partition count.

**Trade-off** (slide 14): caching costs memory. Don't cache enormous DataFrames you'll only use once.


## 2.5 Spark UI tour (no code)

Take three screenshots for your submission:

1. The **Jobs** tab, showing all the jobs you ran in this notebook.
2. The **SQL / DataFrame** tab — click into any query and screenshot its physical plan (the `Scan parquet` node and the operators above it).
3. The **Storage** tab, showing the cached `filtered` DataFrame.

Save them as `notebook1_screenshots/jobs.png`, `notebook1_screenshots/sql.png`, `notebook1_screenshots/storage.png` next to this notebook. You'll include them in your submission zip.


## Closing — stop the session

Free the Spark driver before moving to Notebook 2 (avoids a stale session and a 'port 4040 already bound' on the next session).


In [ ]:
spark.stop()
